# Visualizzazione Risultati Pipeline

Questo notebook permette di recuperare e ispezionare visivamente gli output generati dalla pipeline in produzione (sia riduzione dimensionale pura che clustering), per decidere quali plot integrare nel workflow finale e affiancare i risultati del tuning.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configurazione percorsi
dim_red_dir = Path('../results/lesion/dim_reduction/tsne/23-07_s1.1_p60')
clustering_dir = Path('../results/lesion/dim_reduction_clustering/tsne/kmeans/26-07_s1.1_p30_k5')

# Metadati base e clinici
original_matrix_path = Path('../data/derived/lesion_matrix/21-07_s1.1/matrix.npy')
clinical_meta_path = Path('../assets/metadata/UNIPD_WashU_participants_lesions.tsv')

X_orig = np.load(original_matrix_path)
clinical_meta = pd.read_csv(clinical_meta_path, sep='\t')

%matplotlib inline

## 1. Recupero Embedding da Dim Reduction
In questa sezione recuperiamo lo spazio latente generato dallo step di riduzione dimensionale.

In [ ]:
# Caricamento embedding e metadati
embedding_dr = np.load(dim_red_dir / 'matrix.npy')
metadata_dr = pd.read_csv(dim_red_dir / 'metadata.csv')

# Arricchimento
metadata_dr['volume_voxel'] = X_orig.sum(axis=1)
metadata_dr = metadata_dr.merge(clinical_meta, left_on='subject_id', right_on='participant_id', how='left')

print(f"Embedding Shape: {embedding_dr.shape}")

## 2. Parametri di Configurazione (Dim Reduction)

In [ ]:
# Lettura configurazione se presente
manifest_path = dim_red_dir / 'manifest.json'
if manifest_path.exists():
    with open(manifest_path, 'r') as f:
        conf = json.load(f)
    print("Parametri Dim Reduction:")
    print(json.dumps(conf, indent=2))
else:
    print("File manifest.json non trovato nella cartella.")

## 3. Plotting Modalità (Dim Reduction)
Visualizziamo lo spazio latente colorato per le variabili cliniche e di acquisizione principali, per capire come lo spazio è stato strutturato.

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(16, 12))

# Dataset
sns.scatterplot(x=embedding_dr[:, 0], y=embedding_dr[:, 1], hue=metadata_dr['dataset_x'], palette='Set2', s=25, alpha=0.8, ax=axs[0, 0])
axs[0, 0].set_title('Dataset (Batch Effect)')

# Lato Lesione
sns.scatterplot(x=embedding_dr[:, 0], y=embedding_dr[:, 1], hue=metadata_dr['lesion_side'], palette='Set1', s=25, alpha=0.8, ax=axs[0, 1])
axs[0, 1].set_title('Lato Lesione')

# Volume
scatter = axs[1, 0].scatter(x=embedding_dr[:, 0], y=embedding_dr[:, 1], c=metadata_dr['volume_voxel'], cmap='magma', s=25, alpha=0.8)
axs[1, 0].set_title('Volume Lesionale (Voxel)')
fig.colorbar(scatter, ax=axs[1, 0], label='Voxel Lesionati')

# NIHSS
scatter2 = axs[1, 1].scatter(x=embedding_dr[:, 0], y=embedding_dr[:, 1], c=metadata_dr['NIHSS'], cmap='viridis', s=25, alpha=0.8)
axs[1, 1].set_title('Severità (NIHSS)')
fig.colorbar(scatter2, ax=axs[1, 1], label='Score')

plt.tight_layout()
plt.show()

## 4. Recupero Embedding e Label da Clustering
Carichiamo i risultati ottenuti dallo step successivo della pipeline (Dim Reduction + Clustering).

In [ ]:
embedding_clu = np.load(clustering_dir / 'matrix.npy')
metadata_clu = pd.read_csv(clustering_dir / 'metadata.csv')

print(f"Embedding Shape: {embedding_clu.shape}")
if 'cluster_label' in metadata_clu.columns:
    print("Label trovati:", metadata_clu['cluster_label'].unique())
else:
    print("Nessuna colonna 'cluster_label' trovata nei metadati del clustering.")

## 5. Parametri di Configurazione (Clustering)

In [ ]:
manifest_path_clu = clustering_dir / 'manifest.json'
if manifest_path_clu.exists():
    with open(manifest_path_clu, 'r') as f:
        conf_clu = json.load(f)
    print("Parametri Clustering:")
    print(json.dumps(conf_clu, indent=2))
else:
    print("File manifest.json non trovato nella cartella.")

## 6. Plotting Colorato per Clustering
Visualizziamo la partizione dello spazio generata dall'algoritmo.

In [ ]:
if 'cluster_label' in metadata_clu.columns:
    plt.figure(figsize=(10, 8))
    sns.scatterplot(
        x=embedding_clu[:, 0], 
        y=embedding_clu[:, 1], 
        hue=metadata_clu['cluster_label'], 
        palette='tab10', 
        s=25, 
        alpha=0.9
    )
    plt.title('Spazio Latente - Colorato per Cluster Label')
    plt.legend(title='Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
else:
    print("Impossibile plottare: cluster_label assente.")